In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#extract data ke local colabs
!tar -xzf "/content/drive/MyDrive/Colab Notebooks/Data/liputan6_data.tar.gz" -C "/content/sample_data"


In [61]:
# Load Dataset

import json
import os
import random

train_path = "/content/sample_data/liputan6_data/canonical/train"
dev_path = "/content/sample_data/liputan6_data/canonical/dev"
test_path = "/content/sample_data/liputan6_data/canonical/test"

#fungsi load urutan
def load_all_json(folder):
    data = []

    files = sorted(os.listdir(folder))[:10]

    for filename in files:
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)

            with open(file_path, 'r') as f:
                item = json.load(f)
                data.append(item)

    return data

# fungsi load random
def load_random_json(folder, n=10):
    data = []

    files = [f for f in os.listdir(folder) if f.endswith(".json")]

    sampled_files = random.sample(files, n)  # random tanpa duplikat

    for filename in sampled_files:
        file_path = os.path.join(folder, filename)

        with open(file_path, 'r') as f:
            item = json.load(f)
            data.append(item)

    return data

# dimatikan karena akan menggunakan random, jika ada keperluan silahkan buka
# jika dihidupkan, jangan lupa matikan random dibawah

#train_data = load_all_json(train_path)
#dev_data = load_all_json(dev_path)
#test_data = load_all_json(test_path)

train_data = load_random_json(train_path)
dev_data = load_random_json(dev_path)
test_data = load_random_json(test_path)

print("Jumlah data train_data:", len(train_data))
print("Contoh data train_data:", train_data[0])
print("--"*50)
print("Jumlah data dev_data:", len(dev_data))
print("Contoh data dev_data:", dev_data[0])
print("--"*50)
print("Jumlah data test_data:", len(test_data))
print("Contoh data test_data:", test_data[0])

Jumlah data train_data: 10
Contoh data train_data: {'id': 98240, 'url': 'https://www.liputan6.com/news/read/98240/buya-ismail-meminta-pemecatan-enam-pengurus-dicabut', 'clean_article': [['Liputan6', '.', 'com', ',', 'Jakarta', ':', 'Sesepuh', 'Partai', 'Persatuan', 'Pembangunan', 'Buya', 'Ismail', 'Hasan', 'Metareum', 'mendatangi', 'Kantor', 'Dewan', 'Pimpinan', 'Pusat', 'PPP', 'di', 'Jalan', 'Diponegoro', 'Nomor', '60', ',', 'Jakarta', 'Pusat', ',', 'Kamis', '(', '24/3', ')', '.'], ['Metareum', 'bermaksud', 'bertemu', 'pengurus', 'harian', 'pusat', 'DPP', 'PPP', 'untuk', 'menolak', 'surat', 'pemecatan', 'terhadap', 'enam', 'peserta', 'Silaturahmi', 'Nasional', '(', 'Silatnas', ')', 'Kader', 'PPP', ',', 'awal', 'bulan', 'ini', '[', 'baca', ':', 'Enam', 'Peserta', 'Silatnas', 'PPP', 'Diberhentikan', ']', '.'], ['Metareum', 'datang', 'didampingi', 'pengurus', 'Dewan', 'Pakar', 'DPP', 'PPP', 'dan', 'kuasa', 'hukum', 'enam', 'pengurus', 'DPP', 'yang', 'dipecat', 'oleh', 'Ketua', 'Umum', 'H

In [53]:
# Convert Dataset Untuk mengambil clean_artikel dan clean_summary saja

def convert_to_text(data):
    new_data = []

    for item in data:
        # Gabungkan artikel
        article = " ".join(
            [" ".join(sentence) for sentence in item["clean_article"]]
        )

        # Gabungkan summary
        summary = " ".join(
            [" ".join(sentence) for sentence in item["clean_summary"]]
        )

        # 🔥 ambil extractive summary (index)
        extractive = item["extractive_summary"]

        new_data.append({
            "clean_article": article,
            "clean_summary": summary,
            "extractive_summary": extractive,
        })

    return new_data

train_data_convert = convert_to_text(train_data)
dev_data_convert = convert_to_text(dev_data)
test_data_convert = convert_to_text(test_data)

In [ ]:
train_data_convert

In [19]:
!pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import re

In [55]:
# function lower
def to_lowercase(text):
    return text.lower()

# funtion stopword
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def remove_stopwords(text):
    return stopword_remover.remove(text)


def fix_punctuation(text):
    # hapus spasi sebelum tanda baca
    text = re.sub(r'\s+([.,:;!?])', r'\1', text)

    # rapihin spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def remove_header(text):
    # hapus dari "liputan..." sampai ":" pertama di awal kalimat
    text = re.sub(r'^liputan\S*.*?:\s*', '', text, flags=re.IGNORECASE)
    return text

def remove_baca(text):
    text = re.sub(r'\[\s*baca.*?\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Penggabungan
def clean_text(text):
    text = text.lower()
    text = fix_punctuation(text)
    text = stopword_remover.remove(text)
    text = remove_header(text)
    text = remove_baca(text)
    return text

In [56]:
#Apply cleaning

def apply_cleaning(data):
    new_data = []

    for item in data:
        article = clean_text(item["clean_article"])
        summary = clean_text(item["clean_summary"])

        new_data.append({
            "article": article,
            "summary": summary,
            "extractive_summary": item["extractive_summary"]
        })

    return new_data

train_text_clean = apply_cleaning(train_data_convert)
dev_text_clean = apply_cleaning(dev_data_convert)
test_text_clean = apply_cleaning(test_data_convert)

In [58]:
idx = 9  # ambil data index

print("=== ORIGINAL (convert) ===")
print(train_data_convert[idx]["clean_article"])

print("\n=== CLEANED ===")
print(train_text_clean[idx]["article"])

=== ORIGINAL (convert) ===
Liputan6 . com , Jakarta : Hari ini kondisi kesehatan mantan Presiden Soeharto kembali kritis . Memburuknya keadaan Pak Harto membuat penjagaan di sekitar kediaman keluarga Soeharto , tepatnya di Jalan Cendana Nomor 8-10 , Menteng , Jakarta Pusat , diperketat . Hingga Ahad ( 13/1 ) pukul 18 . 00 WIB , ada sekitar 31 aparat keamanan yang berjaga-jaga di depan rumah Soeharto [ baca : Dokter : Peluang Pak Harto Fifty-Fifty ] . Reporter SCTV Sufiani Tanjung melaporkan , sejak pukul 15 . 00 WIB , pintu pagar rumah keluarga Soeharto ditutup . Gerbang hanya dibuka bila ada pengunjung yang berkepentingan masuk . Tak hanya itu , sejak pukul 16 . 00 WIB , Jalan Cendana disterilkan dari pelintas maupun kendaraan bermotor . Hanya orang-orang tertentu yang diizinkan melewati jalan tersebut , termasuk para wartawan . Seperti halnya di Rumah Sakit Pusat Pertamina , Jakarta Selatan , para peliput baik dari media cetak maupun elektronik memang terus menunggu . Para jurnalis d

In [47]:
from datasets import Dataset

In [63]:
# Convert ke HuggingFace Dataset

train_text_dataset = Dataset.from_list(train_data_convert)
dev_text_dataset   = Dataset.from_list(dev_data_convert)
test_text_dataset  = Dataset.from_list(test_data_convert)

train_clean_dataset = Dataset.from_list(train_text_clean)
dev_clean_dataset   = Dataset.from_list(dev_text_clean)
test_clean_dataset  = Dataset.from_list(test_text_clean)

In [ ]:
print(train_clean_dataset)
print(train_clean_dataset[0:5])

In [ ]:
for i in range(3):
    print(train_text_dataset[i])
    print("---"*50)
    print(train_clean_dataset[i])
    print("===="*50)

In [64]:
# Simpan Ke JSON untuk keperluan upload kaggle

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/train_data_convert.json", "w") as f:
    json.dump(train_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/dev_data_convert.json", "w") as f:
    json.dump(dev_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/test_data_convert.json", "w") as f:
    json.dump(test_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/train_text_clean.json", "w") as f:
    json.dump(train_text_clean, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/dev_text_clean.json", "w") as f:
    json.dump(dev_text_clean, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/test_text_clean.json", "w") as f:
    json.dump(test_text_clean, f)